In [1]:
"""
Run this locally (not in this sandbox -- it can't reach api.deadlock-api.com).

Hits every endpoint that's plausibly relevant to hero/item analysis, pulls ONE
sample response from each, and prints:
  - the endpoint
  - every column/field name it returned
  - one example row so you can see actual values, not just names

Nothing gets merged or cleaned here -- this is purely "show me everything that
exists" so you can decide what's worth keeping.

Requires: pip install requests pandas
"""

import requests
import pandas as pd

BASE = "https://api.deadlock-api.com"
HEADERS = {"User-Agent": "python-requests (deadlock-exploration-script)"}

# path, params -- one small/cheap call per endpoint, just to see the shape
ENDPOINTS = {
    "heroes":               ("/v1/assets/heroes", {}),
    "items":                ("/v1/assets/items", {}),
    "hero-stats":           ("/v1/analytics/hero-stats", {}),
    "item-stats":           ("/v1/analytics/item-stats", {"hero_id": 1}),
    "build-item-stats":     ("/v1/analytics/build-item-stats", {"hero_id": 1}),
    "item-flow-stats":      ("/v1/analytics/item-flow-stats", {"hero_id": 1}),
    "item-permutation-stats": ("/v1/analytics/item-permutation-stats", {"hero_id": 1}),
    "ability-order-stats":  ("/v1/analytics/ability-order-stats", {"hero_id": 1}),
    "hero-ban-stats":       ("/v1/analytics/hero-ban-stats", {}),
    "hero-comb-stats":      ("/v1/analytics/hero-comb-stats", {}),
    "hero-counter-stats":   ("/v1/analytics/hero-counter-stats", {}),
    "hero-synergy-stats":   ("/v1/analytics/hero-synergy-stats", {}),
    "kill-death-stats":     ("/v1/analytics/kill-death-stats", {"hero_id": 1}),
    "player-performance-curve": ("/v1/analytics/player-performance-curve", {"hero_id": 1}),
    "badge-distribution":   ("/v1/analytics/badge-distribution", {}),
    "hero-scoreboard":      ("/v1/analytics/scoreboards/heroes", {"sort_by": "matches"}),
    "player-scoreboard":    ("/v1/analytics/scoreboards/players", {"sort_by": "matches"}),
    "hero-build-stats":     ("/v1/analytics/hero-build-stats/1", {}),
    "lane-matchup-stats":   ("/v1/analytics/lane-matchup-stats", {}),
    "lane-soul-curve":      ("/v1/analytics/lane-soul-curve", {}),
}

for label, (path, params) in ENDPOINTS.items():
    print("=" * 80)
    print(f"{label}   ({path})")
    try:
        r = requests.get(f"{BASE}{path}", params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()

        # some endpoints return a list, some return a dict -- handle both
        sample = data[0] if isinstance(data, list) and len(data) > 0 else data

        if isinstance(sample, dict):
            print(f"columns: {list(sample.keys())}")
            print("example row:")
            print(sample)
        else:
            print("unexpected shape, raw response:")
            print(data)

    except requests.HTTPError as e:
        print(f"FAILED: {e}  (body: {r.text[:300]})")
    except Exception as e:
        print(f"FAILED: {e}")

        print()

heroes   (/v1/assets/heroes)
columns: ['id', 'class_name', 'name', 'description', 'player_selectable', 'disabled', 'in_development', 'needs_testing', 'assigned_players_only', 'tags', 'gun_tag', 'hideout_rich_presence', 'hero_type', 'prerelease_only', 'limited_testing', 'complexity', 'skin', 'images', 'items', 'starting_stats', 'item_slot_info', 'physics', 'colors', 'shop_stat_display', 'cost_bonuses', 'stats_display', 'hero_stats_ui', 'level_info', 'scaling_stats', 'purchase_bonuses', 'standard_level_up_upgrades', 'item_draft_bucketing']
example row:
{'id': 1, 'class_name': 'hero_inferno', 'name': 'Infernus', 'description': {'lore': 'Like most teenagers; Infernus was wild, rebellious, and impetuous.  Unlike most teenagers, Infernus was a creature from another plane and had a supernatural mastery over fire.  Needless to say:  his youth was filled with no small amount of arson, murder, and evidence disposal.  But that was then.  Now an adult, Infernus has mellowed out considerably.  He’s

In [7]:
"""
Run this locally (not in this sandbox -- it can't reach api.deadlock-api.com).

This pulls ONLY what's actually verifiable from the API for Apollo:
  1. Official hero description/role/playstyle text (from /v1/assets/heroes)
  2. Starting stats (from the same endpoint)
  3. A real item BUY-ORDER TIMELINE for Apollo, built from avg_buy_time_relative
     in item-stats -- i.e. what order winning players actually buy things in,
     not a guessed timeline
  4. Laning-phase net worth trajectory for Apollo match-ups, from lane-soul-curve

It deliberately does NOT invent objective timers, camp values, or map
mechanics -- those aren't in this API's schema, and web sources I found
while researching this conflict with each other (some say 3 lanes, some 4;
HP/soul numbers don't agree), so I'm not fabricating false precision there.
Cross-check current objective timers/mechanics against Deadlock's own
in-game tooltips or the official patch notes rather than any single guide site.

Requires: pip install requests pandas
"""

import requests
import pandas as pd
import numpy as np

BASE = "https://api.deadlock-api.com"
HEADERS = {"User-Agent": "python-requests (deadlock-apollo-timeline)"}

TARGET_HERO_ID = 77  # Apollo, resolved in the previous script


def get_json(path, params=None, retries=3):
    for _ in range(retries):
        try:
            r = requests.get(f"{BASE}{path}", params=params or {}, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"  retrying {path}: {e}")
    return None


# ---------------------------------------------------------------------------
# 1 & 2. Official kit text + starting stats
# ---------------------------------------------------------------------------
heroes_raw = get_json("/v1/assets/heroes")
heroes_df = pd.DataFrame(heroes_raw)
apollo_row = heroes_df[heroes_df["id"] == TARGET_HERO_ID].iloc[0]

print("=" * 80)
print("APOLLO -- OFFICIAL KIT TEXT (from the game's own asset data)")
print("=" * 80)
desc = apollo_row["description"]
print(f"Role: {desc.get('role')}")
print(f"Playstyle: {desc.get('playstyle')}")
print(f"Complexity rating: {apollo_row['complexity']}")
print(f"Gun tag: {apollo_row['gun_tag']}")
print(f"Hero type: {apollo_row['hero_type']}")

print("\nStarting stats:")
for stat_name, stat_info in apollo_row["starting_stats"].items():
    print(f"  {stat_name}: {stat_info['value']}")

# ---------------------------------------------------------------------------
# 3. Real buy-order timeline from item-stats avg_buy_time_relative
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("APOLLO -- ACTUAL BUY-ORDER TIMELINE (from real purchase data, not a guess)")
print("=" * 80)

item_stats_raw = get_json("/v1/analytics/item-stats", {"hero_id": TARGET_HERO_ID})
items_raw = get_json("/v1/assets/items")
items_df = pd.DataFrame(items_raw)[["id", "name"]].rename(columns={"id": "item_id", "name": "item_name"})

item_stats_df = pd.DataFrame(item_stats_raw)
item_agg = (
    item_stats_df.groupby("item_id", as_index=False)
    .agg(wins=("wins", "sum"), losses=("losses", "sum"), matches=("matches", "sum"),
         avg_buy_time_relative=("avg_buy_time_relative", "mean"))
)
item_agg["win_rate"] = item_agg["wins"] / (item_agg["wins"] + item_agg["losses"])
item_agg = item_agg.merge(items_df, on="item_id", how="left")
item_agg = item_agg[item_agg["matches"] >= 500].copy()  # reliable sample only

# bucket into rough game phases by relative buy timing (0-100 scale)
def phase(pct):
    if pct < 25:
        return "1. Early (0-25%)"
    elif pct < 50:
        return "2. Early-mid (25-50%)"
    elif pct < 75:
        return "3. Mid-late (50-75%)"
    else:
        return "4. Late (75-100%)"

item_agg["phase"] = item_agg["avg_buy_time_relative"].apply(phase)

for ph in sorted(item_agg["phase"].unique()):
    chunk = item_agg[item_agg["phase"] == ph].sort_values("avg_buy_time_relative")
    print(f"\n--- {ph} ---")
    print(chunk[["item_name", "avg_buy_time_relative", "win_rate", "matches"]]
          .head(10).to_string(index=False))

# ---------------------------------------------------------------------------
# 4. Laning trajectory for Apollo match-ups
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("APOLLO -- LANING NET WORTH TRAJECTORY (real, but small samples -- treat cautiously)")
print("=" * 80)

lane_curve_raw = get_json("/v1/analytics/lane-soul-curve")
lane_curve_df = pd.DataFrame(lane_curve_raw)
apollo_lanes = lane_curve_df[lane_curve_df["hero_ids"].apply(lambda ids: TARGET_HERO_ID in ids)].copy()
apollo_lanes = apollo_lanes[apollo_lanes["matches_played"] >= 50]  # tighter floor than before

if not apollo_lanes.empty:
    for _, row in apollo_lanes.sort_values("matches_played", ascending=False).head(5).iterrows():
        print(f"\nLane {row['hero_ids']} vs {row['enemy_hero_ids']} "
              f"({row['matches_played']} matches):")
        for t, nw in zip(row["sample_times_s"], row["net_worth_diff"]):
            print(f"  t={t}s  net_worth_diff={nw:+.1f}")
else:
    print("No laning data for Apollo cleared the 50-match reliability floor -- "
          "not enough sample to say anything trustworthy about his laning curve specifically.")

print("\n" + "=" * 80)
print("WHAT THIS SCRIPT DELIBERATELY DOES NOT CLAIM")
print("=" * 80)
print("""
No objective timers, jungle camp values, tower HP, or 'at minute X do Y' map
advice is generated here -- that data isn't in this API, and generic web
guides on it contradict each other on basics (lane count, HP values, timers).
Use Apollo's actual buy-order phases above (which ARE real data) combined
with in-game tooltips/patch notes for map mechanics, rather than trusting
any single third-party guide site for exact numbers.
""")

APOLLO -- OFFICIAL KIT TEXT (from the game's own asset data)
Role: None
Playstyle: None
Complexity rating: 2
Gun tag: Spreadshot
Hero type: assassin

Starting stats:
  max_move_speed: 7.2
  sprint_speed: 1.6
  crouch_speed: 4.75
  move_acceleration: 4.0
  light_melee_damage: 63.0
  heavy_melee_damage: 116
  max_health: 770
  weapon_power: 0
  reload_speed: 1
  weapon_power_scale: 1
  proc_build_up_rate_scale: 1
  stamina: 3
  base_health_regen: 1.0
  stamina_regen_per_second: 0.222222
  ability_resource_max: 0
  ability_resource_regen_per_second: 0
  crit_damage_received_scale: 1.0
  tech_duration: 1
  tech_range: 1
  ground_dash_distance_in_meters: 10.0
  ground_dash_duration: 0.68
  air_dash_distance_in_meters: 8.0
  air_dash_duration: 0.47

APOLLO -- ACTUAL BUY-ORDER TIMELINE (from real purchase data, not a guess)

--- 1. Early (0-25%) ---
          item_name  avg_buy_time_relative  win_rate  matches
   Restorative Shot               7.401696  0.506754    13103
       Mystic Burst  